# ElectInfo unifies three polling vendors onto TX-32 precincts

## What this shows
Three polling vendors use three different precinct naming conventions (ALL CAPS, numeric codes, human-readable). ElectInfo joins them onto one authoritative precinct list before any analysis. This is the "dirty-data canonical" — what real cross-source spatial integration looks like.

## Why it matters
Most polling / canvass aggregations live or die on precinct matching. Fuzzy joins with a quality flag beat hard failures that drop mismatched rows silently. The `quality_flag` column here is what downstream dashboards filter on when presenting confidence.

## Prereqs
- `pip install 'siege-utilities[geo]'`
- No credentials.

## Next
- `spatial/03_choropleth_maps.ipynb` renders the unified result as a bivariate choropleth.
- `engines/02_distributed_spark.ipynb` is how this would scale to statewide precinct-level data.


## 1. Synthetic polling output from three vendors

All three vendors are polling the same 8 precincts, but with different naming. `vendor_a` uppercases names; `vendor_b` uses numeric codes; `vendor_c` uses human-readable names with occasional typos. This is exactly the shape ElectInfo sees in practice.

In [1]:
import pandas as pd

vendor_a = pd.DataFrame({
    'precinct_name': ['DOWNTOWN', 'RIVERSIDE', 'EAST SIDE', 'HIGHLANDS',
                      'WESTLAKE', 'LAKESHORE', 'NORTHGATE', 'MIDTOWN'],
    'dem_pct':       [0.62, 0.58, 0.71, 0.48, 0.41, 0.55, 0.52, 0.65],
})
vendor_b = pd.DataFrame({
    'precinct_code': ['TX32-001', 'TX32-002', 'TX32-003', 'TX32-004',
                      'TX32-005', 'TX32-006', 'TX32-007', 'TX32-008'],
    'turnout_pct':   [0.49, 0.51, 0.58, 0.42, 0.39, 0.48, 0.46, 0.55],
})
vendor_c = pd.DataFrame({
    'precinct':        ['Downtown', 'Riversde', 'East Side', 'The Highlands',
                        'Westlake', 'Lake Shore', 'North Gate', 'Midtown'],
    'response_count':  [412, 390, 502, 310, 278, 361, 345, 443],
})
print('vendor_a:', len(vendor_a), 'rows  |  vendor_b:', len(vendor_b), 'rows  |  vendor_c:', len(vendor_c), 'rows')


vendor_a: 8 rows  |  vendor_b: 8 rows  |  vendor_c: 8 rows


## 2. Authoritative precinct list

The canonical 8-precinct list — one row per precinct with every naming convention we need to match against. In production this comes from TIGER 2020 VTDs filtered to TX-32.

In [2]:
authoritative = pd.DataFrame({
    'precinct_id':   [f'TX32-{i:03d}' for i in range(1, 9)],
    'name_upper':    ['DOWNTOWN', 'RIVERSIDE', 'EAST SIDE', 'HIGHLANDS',
                      'WESTLAKE', 'LAKESHORE', 'NORTHGATE', 'MIDTOWN'],
    'name_display':  ['Downtown', 'Riverside', 'East Side', 'Highlands',
                      'Westlake', 'Lakeshore', 'Northgate', 'Midtown'],
})
authoritative


,precinct_id,name_upper,name_display
0,TX32-001,DOWNTOWN,Downtown
1,TX32-002,RIVERSIDE,Riverside
2,TX32-003,EAST SIDE,East Side
3,TX32-004,HIGHLANDS,Highlands
4,TX32-005,WESTLAKE,Westlake
5,TX32-006,LAKESHORE,Lakeshore
6,TX32-007,NORTHGATE,Northgate
7,TX32-008,MIDTOWN,Midtown


## 3. Exact joins for vendor_a and vendor_b

`vendor_a` matches on uppercase name; `vendor_b` matches on the canonical precinct_id. Straightforward merges, and every row should land.

In [3]:
a_joined = authoritative.merge(vendor_a, left_on='name_upper', right_on='precinct_name', how='left')
b_joined = authoritative.merge(vendor_b, left_on='precinct_id', right_on='precinct_code', how='left')
print('vendor_a matched :', a_joined['dem_pct'].notna().sum(), '/', len(authoritative))
print('vendor_b matched :', b_joined['turnout_pct'].notna().sum(), '/', len(authoritative))


vendor_a matched : 8 / 8
vendor_b matched : 8 / 8


## 4. Fuzzy join for vendor_c

Names in `vendor_c` have typos ("Riversde") and spacing variants ("Lake Shore"). `difflib.get_close_matches` is a stdlib fuzzy matcher that handles the typical cases without bringing in a dependency. Low-confidence matches get a `quality_flag` so downstream dashboards can filter them.

In [4]:
from difflib import get_close_matches, SequenceMatcher

def best_match(name, candidates, threshold=0.75):
    matches = get_close_matches(name, candidates, n=1, cutoff=threshold)
    if not matches:
        return None, 0.0
    score = SequenceMatcher(None, name, matches[0]).ratio()
    return matches[0], score

c_out = vendor_c.copy()
c_out[['matched_name', 'match_score']] = c_out['precinct'].apply(
    lambda n: pd.Series(best_match(n, authoritative['name_display'].tolist())),
)
c_out['quality_flag'] = c_out['match_score'].apply(
    lambda s: 'high' if s >= 0.95 else ('medium' if s >= 0.85 else 'low'),
)
c_joined = authoritative.merge(c_out, left_on='name_display', right_on='matched_name', how='left')
c_joined[['precinct_id', 'name_display', 'response_count', 'match_score', 'quality_flag']]


,precinct_id,name_display,response_count,match_score,quality_flag
0,TX32-001,Downtown,412,1.000000,high
1,TX32-002,Riverside,390,0.941176,medium
2,TX32-003,East Side,502,1.000000,high
3,TX32-004,Highlands,310,0.818182,low
4,TX32-005,Westlake,278,1.000000,high
5,TX32-006,Lakeshore,361,0.842105,low
6,TX32-007,Northgate,345,0.842105,low
7,TX32-008,Midtown,443,1.000000,high


## 5. Unified precinct-keyed DataFrame

One row per precinct, three vendor columns, plus the quality flag from the fuzzy join. This is what `spatial/03_choropleth_maps` consumes.

In [5]:
unified = (
    authoritative[['precinct_id', 'name_display']]
    .merge(a_joined[['precinct_id', 'dem_pct']],        on='precinct_id', how='left')
    .merge(b_joined[['precinct_id', 'turnout_pct']],    on='precinct_id', how='left')
    .merge(c_joined[['precinct_id', 'response_count', 'quality_flag']], on='precinct_id', how='left')
)
unified


,precinct_id,name_display,dem_pct,turnout_pct,response_count,quality_flag
0,TX32-001,Downtown,0.62,0.49,412,high
1,TX32-002,Riverside,0.58,0.51,390,medium
2,TX32-003,East Side,0.71,0.58,502,high
3,TX32-004,Highlands,0.48,0.42,310,low
4,TX32-005,Westlake,0.41,0.39,278,high
5,TX32-006,Lakeshore,0.55,0.48,361,low
6,TX32-007,Northgate,0.52,0.46,345,low
7,TX32-008,Midtown,0.65,0.55,443,high


## Related

- **Source**: `siege_utilities/geo/providers/redistricting_data_hub.py` (VTD routing from PR #386), stdlib `difflib` for the fuzzy join.
- **Tests**: `tests/test_boundary_providers*.py`.
- **Next**: `spatial/03_choropleth_maps.ipynb` visualizes this unified DataFrame as a bivariate choropleth over TX-32 precincts.
